# Model Quality Estimation

## Project Overview

In the previous projects, the training and test samples were already prepared. In this project, the focus is different. Here, I want to understand how to design the evaluation process itself, not just train a model and report a score.

## Problem Statement

The main problem in this project is not only training a model, but making sure that the reported model quality is realistic and trustworthy.

A model can produce optimistic results when:
- the data is split incorrectly,
- future information leaks into training,
- repeated groups appear in both training and test sets,
- feature selection is done unfairly,
- or hyperparameters are tuned using information that should not be available.

Because of that, this project is really about learning how to evaluate models correctly and how to choose the right validation strategy for the data.

## Solution 

I will approach this project as an evaluation pipeline design problem rather than a simple modeling exercise. The goal is to build a workflow that produces fair, reproducible, and interpretable results.

My plan is to start from the theoretical foundations, prepare the data carefully, implement and validate the splitting logic, compare it against sklearn, and then extend the pipeline with feature selection and hyperparameter optimization before making a final decision.

## Roadmap

This notebook is organized into the following sections:

1. Theory Questions  
2. Data Preparation  
3. Custom Split Methods  
4. Custom Cross-Validation Methods  
5. Validation Comparison  
6. Feature Selection  
7. Hyperparameter Optimization  
8. Final Comparison and Conclusion

# 1. Theory Questions

## 1.1 What is Leave-One-Out Cross-Validation?

LOOCV is a cross-validation method where I leave out one sample for testing and use all the others for training. If the dataset has \(N\) samples, then the model is trained \(N\) times.

It can be useful for very small datasets because almost all data is used for training each time. But it is expensive, and the result can be unstable since each test fold has only one sample.

## 1.2 How do Grid Search, Randomized Search, and Bayesian Optimization work?

Hyperparameter optimization means finding the best settings for the model before final training.

### Grid Search
Grid Search tries every possible combination from a fixed list of values. It is simple and systematic, but it can be slow.

### Randomized Search
Randomized Search tests only a random subset of combinations. It is usually faster and works well when the search space is large.

### Bayesian Optimization
Bayesian Optimization is more adaptive. It uses previous results to decide what to try next, so it focuses more on promising areas.

In short:
- Grid Search checks all combinations
- Randomized Search checks some random ones
- Bayesian Optimization tries to search smarter

## 1.3 Feature Selection Methods

Feature selection means keeping the useful features and removing the less useful ones. This can improve performance, reduce overfitting, and make the model easier to understand.

The main types are:

### Filter methods
They rank features using statistical measures before training. They are fast and simple.

### Wrapper methods
They test different feature subsets by training the model many times. They are more expensive but often more accurate.

### Embedded methods
They perform feature selection during model training itself.

## 1.4 Pearson Correlation

Pearson correlation measures the linear relationship between a feature and the target. A high positive or negative value means a stronger linear relation, while a value near zero means weak linear relation.

It is useful as a quick filter method, but it only works well for linear patterns.

## 1.5 Chi-Square

Chi-square is used to measure dependence between categorical variables. In feature selection, it is mainly useful when the feature and target are discrete.

It is helpful in classification tasks, but not a general method for every regression problem.

## 1.6 Lasso

Lasso is a linear model with L1 regularization. It can shrink some coefficients to zero, which means it can remove unimportant features automatically.

That is why Lasso is often used as an embedded feature selection method.

## 1.7 Permutation Importance

Permutation importance checks how much the model relies on a feature. If I shuffle one feature and the model performance drops, then that feature was important.

It is easy to understand and based on actual model performance, but it depends on the trained model and can take more time.

## 1.8 SHAP

SHAP explains how features affect predictions. It can show both global feature importance and how each feature changes a specific prediction.

It is very useful for interpretability, but it is more complex than simpler methods like correlation or permutation importance.


## 2. Data preparation

Before comparing validation methods, I first clean the dataset and prepare the features.

The main goal here is:
- remove columns that are not useful for modeling
- handle missing values
- create a cleaner feature set for the next steps

In [1]:
import numpy as np
import pandas as pd
from collections import Counter

df = pd.read_json("data/train.json")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData info:")
print(df.info())

print("\nFirst rows:")
display(df.head())

print("\nSample values from the 'features' column:")
display(df["features"].head())

Shape: (49352, 15)

Columns:
['bathrooms', 'bedrooms', 'building_id', 'created', 'description', 'display_address', 'features', 'latitude', 'listing_id', 'longitude', 'manager_id', 'photos', 'price', 'street_address', 'interest_level']

Data info:
<class 'pandas.core.frame.DataFrame'>
Index: 49352 entries, 4 to 124009
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   bathrooms        49352 non-null  float64
 1   bedrooms         49352 non-null  int64  
 2   building_id      49352 non-null  object 
 3   created          49352 non-null  object 
 4   description      49352 non-null  object 
 5   display_address  49352 non-null  object 
 6   features         49352 non-null  object 
 7   latitude         49352 non-null  float64
 8   listing_id       49352 non-null  int64  
 9   longitude        49352 non-null  float64
 10  manager_id       49352 non-null  object 
 11  photos           49352 non-null  object 
 12 

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low



Sample values from the 'features' column:


4     [Dining Room, Pre-War, Laundry in Building, Di...
6     [Doorman, Elevator, Laundry in Building, Dishw...
9     [Doorman, Elevator, Laundry in Building, Laund...
10                                                   []
15    [Doorman, Elevator, Fitness Center, Laundry in...
Name: features, dtype: object

## Creating the required binary features

The `features` column contains raw apartment attributes stored as text lists. To use them in modeling, I need to normalize their names first, then convert the required values into binary columns.

In [2]:
required_features = [
    'Elevator',
    'HardwoodFloors',
    'CatsAllowed',
    'DogsAllowed',
    'Doorman',
    'Dishwasher',
    'NoFee',
    'LaundryinBuilding',
    'FitnessCenter',
    'Pre-War',
    'LaundryinUnit',
    'RoofDeck',
    'OutdoorSpace',
    'DiningRoom',
    'HighSpeedInternet',
    'Balcony',
    'SwimmingPool',
    'LaundryInBuilding',
    'NewConstruction',
    'Terrace'
]

def normalize_feature_name(feature):
    return str(feature).replace("-", "").replace(" ", "").strip()

df["features_clean"] = df["features"].apply(
    lambda values: [normalize_feature_name(v) for v in values]
)

for feature in required_features:
    feature_norm = normalize_feature_name(feature)
    df[feature] = df["features_clean"].apply(lambda values: int(feature_norm in values))

feature_list = ["bathrooms", "bedrooms"] + required_features
model_df = df[feature_list + ["price"]].copy()

print("Number of required binary features:", len(required_features))
print("Number of modeling features:", len(feature_list))
print("\nPreview of created binary features:")
display(df[required_features].head())

print("\nSample of original vs cleaned features:")
display(df[["features", "features_clean"]].head())

print("\nModel dataframe shape:", model_df.shape)

Number of required binary features: 20
Number of modeling features: 22

Preview of created binary features:


,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,FitnessCenter,Pre-War,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
4,0,1,1,1,0,1,0,1,0,1,0,0,0,1,0,0,0,0,0,0
6,1,1,0,0,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0
9,1,1,0,0,1,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0
10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
15,1,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0



Sample of original vs cleaned features:


,features,features_clean
4,"[Dining Room, Pre-War, Laundry in Building, Di...","[DiningRoom, PreWar, LaundryinBuilding, Dishwa..."
6,"[Doorman, Elevator, Laundry in Building, Dishw...","[Doorman, Elevator, LaundryinBuilding, Dishwas..."
9,"[Doorman, Elevator, Laundry in Building, Laund...","[Doorman, Elevator, LaundryinBuilding, Laundry..."
10,[],[]
15,"[Doorman, Elevator, Fitness Center, Laundry in...","[Doorman, Elevator, FitnessCenter, LaundryinBu..."



Model dataframe shape: (49352, 23)


## Converting the time column

Since some validation methods depend on time order, I first convert the `created` column to datetime format.

In [3]:
df["created"] = pd.to_datetime(df["created"])

print(df["created"].dtype)
display(df[["created"]].head())

datetime64[ns]


,created
4,2016-06-16 05:55:27
6,2016-06-01 05:44:33
9,2016-06-14 15:19:59
10,2016-06-24 07:54:24
15,2016-06-28 03:50:23


## 3.1 Random train/test split

I will start with the simplest split method: a random train/test split controlled by `test_size`. This method is appropriate when the data does not need time-aware splitting.

In [4]:
def random_train_test_split(df, test_size=0.2, random_state=42):
    if not 0 < test_size < 1:
        raise ValueError("test_size must be between 0 and 1")

    np.random.seed(random_state)

    indices = np.arange(len(df))
    np.random.shuffle(indices)

    test_count = int(len(df) * test_size)

    test_indices = indices[:test_count]
    train_indices = indices[test_count:]

    train_df = df.iloc[train_indices].copy()
    test_df = df.iloc[test_indices].copy()

    return train_df, test_df


train_df, test_df = random_train_test_split(model_df, test_size=0.2, random_state=42)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Total rows preserved:", len(train_df) + len(test_df) == len(model_df))
print("No overlap between train and test:", len(set(train_df.index).intersection(set(test_df.index))) == 0)



Train shape: (39482, 23)
Test shape: (9870, 23)
Total rows preserved: True
No overlap between train and test: True


## 3.2 Random train/validation/test split

The next step is to split the dataset into three random parts: training, validation, and test. This is useful when I want to keep a separate validation set for model selection, feature selection, or hyperparameter tuning.

In [5]:
def random_train_val_test_split(df, validation_size=0.2, test_size=0.2, random_state=42):
    if not 0 < validation_size < 1:
        raise ValueError("validation_size must be between 0 and 1")
    if not 0 < test_size < 1:
        raise ValueError("test_size must be between 0 and 1")
    if validation_size + test_size >= 1:
        raise ValueError("validation_size + test_size must be less than 1")

    np.random.seed(random_state)

    indices = np.arange(len(df))
    np.random.shuffle(indices)

    test_count = int(len(df) * test_size)
    val_count = int(len(df) * validation_size)

    test_indices = indices[:test_count]
    val_indices = indices[test_count:test_count + val_count]
    train_indices = indices[test_count + val_count:]

    train_df = df.iloc[train_indices].copy()
    val_df = df.iloc[val_indices].copy()
    test_df = df.iloc[test_indices].copy()

    return train_df, val_df, test_df


train_df_3, val_df_3, test_df_3 = random_train_val_test_split(
    model_df,
    validation_size=0.2,
    test_size=0.2,
    random_state=42
)

print("Train shape:", train_df_3.shape)
print("Validation shape:", val_df_3.shape)
print("Test shape:", test_df_3.shape)

total_ok = len(train_df_3) + len(val_df_3) + len(test_df_3) == len(model_df)
print("Total rows preserved:", total_ok)

all_overlap = (
    len(set(train_df_3.index).intersection(set(val_df_3.index))) == 0 and
    len(set(train_df_3.index).intersection(set(test_df_3.index))) == 0 and
    len(set(val_df_3.index).intersection(set(test_df_3.index))) == 0
)
print("No overlap across splits:", all_overlap)

Train shape: (29612, 23)
Validation shape: (9870, 23)
Test shape: (9870, 23)
Total rows preserved: True
No overlap across splits: True


## 3.3 Date-based train/test split

Now I will implement a time-aware train/test split using a date threshold. This method is more appropriate when the data has a time relationship and I want to avoid using future information during training.

In [6]:
def date_train_test_split(df, date_col="created", date_split="2016-06-15"):
    if date_col not in df.columns:
        raise ValueError(f"Column '{date_col}' not found in dataframe")

    split_date = pd.to_datetime(date_split)

    train_df = df[df[date_col] < split_date].copy()
    test_df = df[df[date_col] >= split_date].copy()

    return train_df, test_df


train_time_df, test_time_df = date_train_test_split(df, date_col="created", date_split="2016-06-15")

print("Train shape:", train_time_df.shape)
print("Test shape:", test_time_df.shape)

print("Total rows preserved:", len(train_time_df) + len(test_time_df) == len(df))
print("Latest train date:", train_time_df["created"].max())
print("Earliest test date:", test_time_df["created"].min())
print("Time order respected:", train_time_df["created"].max() < test_time_df["created"].min())

Train shape: (40991, 36)
Test shape: (8361, 36)
Total rows preserved: True
Latest train date: 2016-06-14 22:51:35
Earliest test date: 2016-06-15 01:10:37
Time order respected: True


## 3.4 Date-based train/validation/test split

Here I extend the time-aware split into three parts: training, validation, and test. This allows me to keep the whole evaluation process consistent with the time order of the data.

In [7]:
def date_train_val_test_split(df, date_col="created", validation_date="2016-06-10", test_date="2016-06-20"):
    if date_col not in df.columns:
        raise ValueError(f"Column '{date_col}' not found in dataframe")

    validation_date = pd.to_datetime(validation_date)
    test_date = pd.to_datetime(test_date)

    if validation_date >= test_date:
        raise ValueError("validation_date must be earlier than test_date")

    train_df = df[df[date_col] < validation_date].copy()
    val_df = df[(df[date_col] >= validation_date) & (df[date_col] < test_date)].copy()
    test_df = df[df[date_col] >= test_date].copy()

    return train_df, val_df, test_df


train_time_3, val_time_3, test_time_3 = date_train_val_test_split(
    df,
    date_col="created",
    validation_date="2016-06-10",
    test_date="2016-06-20"
)

print("Train shape:", train_time_3.shape)
print("Validation shape:", val_time_3.shape)
print("Test shape:", test_time_3.shape)

print("Total rows preserved:", len(train_time_3) + len(val_time_3) + len(test_time_3) == len(df))

no_overlap = (
    len(set(train_time_3.index).intersection(set(val_time_3.index))) == 0 and
    len(set(train_time_3.index).intersection(set(test_time_3.index))) == 0 and
    len(set(val_time_3.index).intersection(set(test_time_3.index))) == 0
)
print("No overlap across splits:", no_overlap)

print("Latest train date:", train_time_3["created"].max())
print("Earliest validation date:", val_time_3["created"].min())
print("Latest validation date:", val_time_3["created"].max())
print("Earliest test date:", test_time_3["created"].min())

time_order_ok = (
    train_time_3["created"].max() < val_time_3["created"].min() and
    val_time_3["created"].max() < test_time_3["created"].min()
) 

print("Time order respected:", time_order_ok)

Train shape: (37793, 36)
Validation shape: (5563, 36)
Test shape: (5996, 36)
Total rows preserved: True
No overlap across splits: True
Latest train date: 2016-06-09 23:42:21
Earliest validation date: 2016-06-10 01:01:13
Latest validation date: 2016-06-19 21:57:21
Earliest test date: 2016-06-20 10:00:03
Time order respected: True


## 3.5 Deterministic splitting

A deterministic split means that the same input data and the same configuration should always produce the same split.

In the random split functions, I made the procedure deterministic by using a fixed `random_state`. This is important because it makes the results reproducible and allows fair comparisons between different models or experiments.

For the date-based splits, the procedure is already deterministic because the split depends directly on fixed date thresholds rather than random sampling.

# 4. Custom Cross-Validation Methods

After implementing the basic split methods, the next step is to implement cross-validation methods manually. Unlike a single holdout split, cross-validation evaluates the model across multiple splits, which usually gives a more stable view of model performance.

## 4.1 K-Fold

In [8]:
def custom_kfold_indices(df, k=5, random_state=42, shuffle=True):
    if k < 2:
        raise ValueError("k must be at least 2")

    indices = np.arange(len(df))

    if shuffle:
        np.random.seed(random_state)
        np.random.shuffle(indices)

    folds = np.array_split(indices, k)
    split_indices = []

    for i in range(k):
        test_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        split_indices.append((train_idx, test_idx))

    return split_indices


kfold_splits = custom_kfold_indices(model_df, k=5, random_state=42, shuffle=True)

print("Number of folds:", len(kfold_splits))

for i, (train_idx, test_idx) in enumerate(kfold_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")



Number of folds: 5
Fold 1: train=39481, test=9871
Fold 2: train=39481, test=9871
Fold 3: train=39482, test=9870
Fold 4: train=39482, test=9870
Fold 5: train=39482, test=9870


## Grouped K-Fold

Here I split the data by `manager_id` so the same manager does not appear in both train and test.

In [9]:
def custom_group_kfold_indices(df, group_field, k=5, random_state=42, shuffle=True):
    if k < 2:
        raise ValueError("k must be at least 2")
    if group_field not in df.columns:
        raise ValueError(f"Column '{group_field}' not found in dataframe")

    unique_groups = df[group_field].dropna().unique()

    if shuffle:
        np.random.seed(random_state)
        np.random.shuffle(unique_groups)

    group_folds = np.array_split(unique_groups, k)
    split_indices = []

    for i in range(k):
        test_groups = set(group_folds[i])
        test_mask = df[group_field].isin(test_groups)
        train_mask = ~test_mask

        train_idx = df[train_mask].index.to_numpy()
        test_idx = df[test_mask].index.to_numpy()

        split_indices.append((train_idx, test_idx))

    return split_indices


group_kfold_splits = custom_group_kfold_indices(df, group_field="manager_id", k=5, random_state=42, shuffle=True)

print("Number of folds:", len(group_kfold_splits))

for i, (train_idx, test_idx) in enumerate(group_kfold_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

Number of folds: 5
Fold 1: train=39400, test=9952
Fold 2: train=36872, test=12480
Fold 3: train=40065, test=9287
Fold 4: train=40795, test=8557
Fold 5: train=40276, test=9076


## Checking group overlap

Here I verify that the same `manager_id` does not appear in both train and test within the same fold.

In [10]:
for i, (train_idx, test_idx) in enumerate(group_kfold_splits, start=1):
    train_groups = set(df.loc[train_idx, "manager_id"])
    test_groups = set(df.loc[test_idx, "manager_id"])
    overlap = train_groups.intersection(test_groups)

    print(f"Fold {i}: overlapping groups = {len(overlap)}")

Fold 1: overlapping groups = 0
Fold 2: overlapping groups = 0
Fold 3: overlapping groups = 0
Fold 4: overlapping groups = 0
Fold 5: overlapping groups = 0


## Stratified K-Fold

In this split, I try to keep the target distribution similar across folds.  
Since `price` is continuous, I first convert it into bins and then use those bins for stratification.

In [11]:
def custom_stratified_kfold_indices(df, stratify_field, k=5, random_state=42, shuffle=True):
    if k < 2:
        raise ValueError("k must be at least 2")
    if stratify_field not in df.columns:
        raise ValueError(f"Column '{stratify_field}' not found in dataframe")

    indices = np.arange(len(df))
    labels = df[stratify_field].to_numpy()

    if shuffle:
        np.random.seed(random_state)

    split_indices = []
    test_folds = [[] for _ in range(k)]

    for label in np.unique(labels):
        label_indices = indices[labels == label]

        if shuffle:
            np.random.shuffle(label_indices)

        label_folds = np.array_split(label_indices, k)

        for i in range(k):
            test_folds[i].extend(label_folds[i])

    for i in range(k):
        test_idx = np.array(test_folds[i])
        train_idx = np.setdiff1d(indices, test_idx)
        split_indices.append((train_idx, test_idx))

    return split_indices


df["price_bin"] = pd.qcut(df["price"], q=5, labels=False, duplicates="drop")

stratified_splits = custom_stratified_kfold_indices(
    df,
    stratify_field="price_bin",
    k=5,
    random_state=42,
    shuffle=True
)

print("Number of folds:", len(stratified_splits))

for i, (train_idx, test_idx) in enumerate(stratified_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

Number of folds: 5
Fold 1: train=39479, test=9873
Fold 2: train=39480, test=9872
Fold 3: train=39482, test=9870
Fold 4: train=39483, test=9869
Fold 5: train=39484, test=9868


## Checking the bin distribution

Here I check whether the `price_bin` distribution is similar across the test folds.  
If the split is properly stratified, the proportions should look close from one fold to another.

In [12]:
for i, (_, test_idx) in enumerate(stratified_splits, start=1):
    distribution = df.iloc[test_idx]["price_bin"].value_counts(normalize=True).sort_index()
    print(f"\nFold {i} price_bin distribution:")
    print(distribution)


Fold 1 price_bin distribution:
price_bin
0    0.202471
1    0.199129
2    0.202168
3    0.201459
4    0.194774
Name: proportion, dtype: float64

Fold 2 price_bin distribution:
price_bin
0    0.202492
1    0.199149
2    0.202087
3    0.201479
4    0.194793
Name: proportion, dtype: float64

Fold 3 price_bin distribution:
price_bin
0    0.202533
1    0.199088
2    0.202128
3    0.201520
4    0.194732
Name: proportion, dtype: float64

Fold 4 price_bin distribution:
price_bin
0    0.202553
1    0.199108
2    0.202148
3    0.201439
4    0.194751
Name: proportion, dtype: float64

Fold 5 price_bin distribution:
price_bin
0    0.202473
1    0.199128
2    0.202169
3    0.201459
4    0.194771
Name: proportion, dtype: float64


## Time Series split

For time-based validation, I sort the data by `created` and split it in chronological order.  
This way, each training set contains only past data and each test set contains future data.

In [13]:
def custom_time_series_split_indices(df, date_field, k=5):
    if k < 2:
        raise ValueError("k must be at least 2")
    if date_field not in df.columns:
        raise ValueError(f"Column '{date_field}' not found in dataframe")

    df_sorted = df.sort_values(date_field).reset_index(drop=False)
    n = len(df_sorted)

    fold_size = n // k
    split_indices = []

    for i in range(1, k):
        train_end = i * fold_size
        test_end = (i + 1) * fold_size if i < k - 1 else n

        train_idx = df_sorted.loc[:train_end - 1, "index"].to_numpy()
        test_idx = df_sorted.loc[train_end:test_end - 1, "index"].to_numpy()

        split_indices.append((train_idx, test_idx))

    return split_indices


time_splits = custom_time_series_split_indices(df, date_field="created", k=5)

print("Number of splits:", len(time_splits))

for i, (train_idx, test_idx) in enumerate(time_splits, start=1):
    print(f"Split {i}: train={len(train_idx)}, test={len(test_idx)}")

Number of splits: 4
Split 1: train=9870, test=9870
Split 2: train=19740, test=9870
Split 3: train=29610, test=9870
Split 4: train=39480, test=9872


## Checking the time order

Here I verify that each test split comes strictly after its corresponding training split.  
If the split is correct, the latest training date should always be earlier than the earliest test date.

In [14]:
for i, (train_idx, test_idx) in enumerate(time_splits, start=1):
    train_dates = df.loc[train_idx, "created"]
    test_dates = df.loc[test_idx, "created"]

    print(f"\nSplit {i}")
    print("Latest train date:", train_dates.max())
    print("Earliest test date:", test_dates.min())
    print("Time order respected:", train_dates.max() < test_dates.min())


Split 1
Latest train date: 2016-04-19 03:56:39
Earliest test date: 2016-04-19 03:57:40
Time order respected: True

Split 2
Latest train date: 2016-05-07 03:11:17
Earliest test date: 2016-05-07 03:11:44
Time order respected: True

Split 3
Latest train date: 2016-05-24 22:30:51
Earliest test date: 2016-05-25 01:10:42
Time order respected: True

Split 4
Latest train date: 2016-06-12 08:16:40
Earliest test date: 2016-06-12 08:17:43
Time order respected: True


## Comparing with sklearn K-Fold

To make sure my custom implementation behaves as expected, I compare it with the built-in `KFold` from sklearn using the same settings.

In [15]:
from sklearn.model_selection import KFold

sklearn_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
sklearn_kfold_splits = list(sklearn_kfold.split(model_df))

print("Custom K-Fold:")
for i, (train_idx, test_idx) in enumerate(kfold_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

print("\nSklearn K-Fold:")
for i, (train_idx, test_idx) in enumerate(sklearn_kfold_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

Custom K-Fold:
Fold 1: train=39481, test=9871
Fold 2: train=39481, test=9871
Fold 3: train=39482, test=9870
Fold 4: train=39482, test=9870
Fold 5: train=39482, test=9870

Sklearn K-Fold:
Fold 1: train=39481, test=9871
Fold 2: train=39481, test=9871
Fold 3: train=39482, test=9870
Fold 4: train=39482, test=9870
Fold 5: train=39482, test=9870


## Comparing fold statistics

Here I compare the first custom fold with the first sklearn fold using a few selected features.  
If both implementations are similar, their training-set averages should also be close.

In [16]:
custom_train_idx, custom_test_idx = kfold_splits[0]
sklearn_train_idx, sklearn_test_idx = sklearn_kfold_splits[0]

selected_features = ["bathrooms", "bedrooms", "Elevator", "Doorman", "Dishwasher"]

custom_train_stats = model_df.iloc[custom_train_idx][selected_features].mean().rename("custom_train_mean")
sklearn_train_stats = model_df.iloc[sklearn_train_idx][selected_features].mean().rename("sklearn_train_mean")

comparison_df = pd.concat([custom_train_stats, sklearn_train_stats], axis=1)
comparison_df["absolute_diff"] = (comparison_df["custom_train_mean"] - comparison_df["sklearn_train_mean"]).abs()

comparison_df

,custom_train_mean,sklearn_train_mean,absolute_diff
bathrooms,1.212773,1.212773,0.0
bedrooms,1.538462,1.538462,0.0
Elevator,0.525493,0.525493,0.0
Doorman,0.424407,0.424407,0.0
Dishwasher,0.413085,0.413085,0.0


The comparison shows that the custom K-Fold implementation produces the same training distribution as sklearn for the selected features in the first fold. This supports the correctness of the custom implementation, not only in terms of fold sizes, but also in terms of feature distribution inside the training split.

## Comparing with sklearn Group K-Fold

To validate my grouped split implementation, I compare it with `GroupKFold` from sklearn using the same grouping variable.

In [17]:
from sklearn.model_selection import GroupKFold

sklearn_group_kfold = GroupKFold(n_splits=5)
sklearn_group_splits = list(sklearn_group_kfold.split(df, groups=df["manager_id"]))

print("Custom Group K-Fold:")
for i, (train_idx, test_idx) in enumerate(group_kfold_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

print("\nSklearn Group K-Fold:")
for i, (train_idx, test_idx) in enumerate(sklearn_group_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

Custom Group K-Fold:
Fold 1: train=39400, test=9952
Fold 2: train=36872, test=12480
Fold 3: train=40065, test=9287
Fold 4: train=40795, test=8557
Fold 5: train=40276, test=9076

Sklearn Group K-Fold:
Fold 1: train=39481, test=9871
Fold 2: train=39481, test=9871
Fold 3: train=39482, test=9870
Fold 4: train=39482, test=9870
Fold 5: train=39482, test=9870


## Checking group separation

Here I verify that neither the custom split nor the sklearn split places the same `manager_id` in both train and test within the same fold.

In [18]:
for i, ((custom_train_idx, custom_test_idx), (sk_train_idx, sk_test_idx)) in enumerate(
    zip(group_kfold_splits, sklearn_group_splits), start=1
):
    custom_overlap = set(df.loc[custom_train_idx, "manager_id"]).intersection(set(df.loc[custom_test_idx, "manager_id"]))
    sklearn_overlap = set(df.iloc[sk_train_idx]["manager_id"]).intersection(set(df.iloc[sk_test_idx]["manager_id"]))

    print(f"Fold {i}: custom_overlap={len(custom_overlap)}, sklearn_overlap={len(sklearn_overlap)}")

Fold 1: custom_overlap=0, sklearn_overlap=0
Fold 2: custom_overlap=0, sklearn_overlap=0
Fold 3: custom_overlap=0, sklearn_overlap=0
Fold 4: custom_overlap=0, sklearn_overlap=0
Fold 5: custom_overlap=0, sklearn_overlap=0


The custom and sklearn implementations both satisfy the main goal of Group K-Fold: the same group does not appear in both training and test sets within the same fold.

However, the fold sizes are more uneven in the custom implementation. This happens because my version splits the groups directly, while sklearn uses a more balanced strategy to keep the number of samples per fold closer across the splits.

## Comparing with sklearn Stratified K-Fold

To check my stratified split implementation, I compare it with sklearn’s `StratifiedKFold` using the same binned target.

In [19]:
from sklearn.model_selection import StratifiedKFold

sklearn_stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
sklearn_stratified_splits = list(sklearn_stratified.split(df, df["price_bin"]))

print("Custom Stratified K-Fold:")
for i, (train_idx, test_idx) in enumerate(stratified_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

print("\nSklearn Stratified K-Fold:")
for i, (train_idx, test_idx) in enumerate(sklearn_stratified_splits, start=1):
    print(f"Fold {i}: train={len(train_idx)}, test={len(test_idx)}")

Custom Stratified K-Fold:
Fold 1: train=39479, test=9873
Fold 2: train=39480, test=9872
Fold 3: train=39482, test=9870
Fold 4: train=39483, test=9869
Fold 5: train=39484, test=9868

Sklearn Stratified K-Fold:
Fold 1: train=39481, test=9871
Fold 2: train=39481, test=9871
Fold 3: train=39482, test=9870
Fold 4: train=39482, test=9870
Fold 5: train=39482, test=9870


## Comparing the bin distributions

Here I compare the `price_bin` distribution in the test folds for both the custom and sklearn stratified splits.  
If both methods work properly, the proportions should look very similar across folds.

In [20]:
print("Custom Stratified K-Fold distributions:")
for i, (_, test_idx) in enumerate(stratified_splits, start=1):
    distribution = df.iloc[test_idx]["price_bin"].value_counts(normalize=True).sort_index()
    print(f"\nFold {i}")
    print(distribution)

print("\nSklearn Stratified K-Fold distributions:")
for i, (_, test_idx) in enumerate(sklearn_stratified_splits, start=1):
    distribution = df.iloc[test_idx]["price_bin"].value_counts(normalize=True).sort_index()
    print(f"\nFold {i}")
    print(distribution)

Custom Stratified K-Fold distributions:

Fold 1
price_bin
0    0.202471
1    0.199129
2    0.202168
3    0.201459
4    0.194774
Name: proportion, dtype: float64

Fold 2
price_bin
0    0.202492
1    0.199149
2    0.202087
3    0.201479
4    0.194793
Name: proportion, dtype: float64

Fold 3
price_bin
0    0.202533
1    0.199088
2    0.202128
3    0.201520
4    0.194732
Name: proportion, dtype: float64

Fold 4
price_bin
0    0.202553
1    0.199108
2    0.202148
3    0.201439
4    0.194751
Name: proportion, dtype: float64

Fold 5
price_bin
0    0.202473
1    0.199128
2    0.202169
3    0.201459
4    0.194771
Name: proportion, dtype: float64

Sklearn Stratified K-Fold distributions:

Fold 1
price_bin
0    0.202512
1    0.199169
2    0.202208
3    0.201398
4    0.194712
Name: proportion, dtype: float64

Fold 2
price_bin
0    0.202512
1    0.199169
2    0.202107
3    0.201398
4    0.194813
Name: proportion, dtype: float64

Fold 3
price_bin
0    0.202432
1    0.199088
2    0.202128
3    0.2015

The custom Stratified K-Fold implementation preserved the `price_bin` distribution across folds quite well. The resulting distributions were also very close to sklearn’s implementation, which supports the correctness of the custom method.

## Choosing the most appropriate validation scheme

After comparing the implemented validation methods, I do not think there is one best choice for every case. The right validation scheme depends on the dataset and on the type of leakage that should be avoided.

- Standard K-Fold is a good general-purpose option when observations are independent.
- Group K-Fold is more suitable when the same entity can appear multiple times and group leakage is possible.
- Stratified K-Fold is helpful when I want to keep the target distribution balanced across folds.
- Time-based splitting and TimeSeriesSplit are more appropriate when the data has a chronological structure.

For this dataset, I consider time-based validation the most realistic choice. Since the `created` column is available, using time-aware splitting makes it possible to simulate real future prediction more faithfully. It preserves the chronological order of the data and avoids training on future information.

## Feature Selection

After selecting the validation strategy, I move to feature selection.  
The goal is to identify a smaller subset of useful features while preserving most of the predictive signal.

I start with Lasso, which is an embedded feature selection method. Since time-based validation was chosen as the most appropriate scheme, I keep the chronological order of the data and split it into train, validation, and test sets using the `created` column.

In [21]:
df = df.sort_values("created").reset_index(drop=True)

n = len(df)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_df_fs = df.iloc[:train_end].copy()
val_df_fs = df.iloc[train_end:val_end].copy()
test_df_fs = df.iloc[val_end:].copy()

print("Train shape:", train_df_fs.shape)
print("Validation shape:", val_df_fs.shape)
print("Test shape:", test_df_fs.shape)

print("Latest train date:", train_df_fs["created"].max())
print("Earliest validation date:", val_df_fs["created"].min())
print("Latest validation date:", val_df_fs["created"].max())
print("Earliest test date:", test_df_fs["created"].min())

Train shape: (29611, 37)
Validation shape: (9870, 37)
Test shape: (9871, 37)
Latest train date: 2016-05-25 01:10:42
Earliest validation date: 2016-05-25 01:11:44
Latest validation date: 2016-06-12 08:17:43
Earliest test date: 2016-06-12 08:18:50


## Time-based split for feature selection

To stay consistent with the chosen validation strategy, I split the data in chronological order.  
I use the oldest 60% for training, the next 20% for validation, and the most recent 20% for testing.

In [22]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error

X_train = train_df_fs[feature_list]
y_train = train_df_fs["price"]

X_val = val_df_fs[feature_list]
y_val = val_df_fs["price"]

X_test = test_df_fs[feature_list]
y_test = test_df_fs["price"]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

lasso = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso.fit(X_train_scaled, y_train)

val_pred = lasso.predict(X_val_scaled)

print("Validation MAE:", mean_absolute_error(y_val, val_pred))
print("Validation MSE:", mean_squared_error(y_val, val_pred))

Validation MAE: 922.2138133216474
Validation MSE: 3999291.5479598134


## Lasso as an embedded feature selection method

In this step, I train a Lasso model on the scaled training data and evaluate it on the validation set.  
Lasso is useful because it can shrink weak feature coefficients toward zero, which makes it a practical method for feature selection.

In [23]:
lasso_importance = pd.Series(np.abs(lasso.coef_), index=feature_list).sort_values(ascending=False)

print("Lasso feature importance ranking:")
display(lasso_importance)

top_10_lasso_features = lasso_importance.head(10).index.tolist()

print("Top 10 features selected by Lasso:")
print(top_10_lasso_features)

Lasso feature importance ranking:


bathrooms            1197.356738
Doorman               563.976006
bedrooms              456.110313
LaundryinUnit         215.873329
LaundryinBuilding     188.353791
DogsAllowed           181.255971
NoFee                 131.868958
Terrace               125.069570
LaundryInBuilding     113.279833
HardwoodFloors        104.625855
Elevator               96.511263
HighSpeedInternet      94.864739
CatsAllowed            78.619633
Pre-War                72.792170
DiningRoom             57.500481
Balcony                57.123525
OutdoorSpace           57.048532
FitnessCenter          45.527376
RoofDeck               27.126764
SwimmingPool            5.118130
NewConstruction         2.923651
Dishwasher              0.070717
dtype: float64

Top 10 features selected by Lasso:
['bathrooms', 'Doorman', 'bedrooms', 'LaundryinUnit', 'LaundryinBuilding', 'DogsAllowed', 'NoFee', 'Terrace', 'LaundryInBuilding', 'HardwoodFloors']


## Ranking features with Lasso

To estimate feature importance, I use the absolute values of the learned Lasso coefficients.  
Features with larger coefficient magnitudes are treated as more influential for the model.

In [24]:
X_train_top10 = train_df_fs[top_10_lasso_features]
X_val_top10 = val_df_fs[top_10_lasso_features]
X_test_top10 = test_df_fs[top_10_lasso_features]

scaler_top10 = StandardScaler()

X_train_top10_scaled = scaler_top10.fit_transform(X_train_top10)
X_val_top10_scaled = scaler_top10.transform(X_val_top10)
X_test_top10_scaled = scaler_top10.transform(X_test_top10)

lasso_top10 = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso_top10.fit(X_train_top10_scaled, y_train)

val_pred_top10 = lasso_top10.predict(X_val_top10_scaled)

print("Validation MAE with top 10 Lasso features:", mean_absolute_error(y_val, val_pred_top10))
print("Validation MSE with top 10 Lasso features:", mean_squared_error(y_val, val_pred_top10))

Validation MAE with top 10 Lasso features: 918.2332725081059
Validation MSE with top 10 Lasso features: 3997488.7501502866


## Comparing the full feature set with the top 10 Lasso features

After ranking the features, I retrain the model using only the top 10 selected by Lasso.  
This helps me check whether a smaller feature subset can keep most of the predictive performance.

The top 10 Lasso-selected features produced validation results that were very close to the full feature set.  
This suggests that a smaller subset of features can preserve most of the useful predictive signal while making the model more compact.

## Simple feature selection with nan-ratio and correlation

Next, I apply a simpler filter-based approach.  
I rank the features using two criteria: the amount of missing data and their correlation with the target.  
Then I keep the top 10 features and compare the validation quality again.

In [25]:
candidate_features = feature_list.copy()

nan_ratio = train_df_fs[candidate_features].isna().mean()

correlation = train_df_fs[candidate_features + ["price"]].corr()["price"].drop("price").abs()

simple_score_df = pd.DataFrame({
    "nan_ratio": nan_ratio,
    "abs_corr_with_price": correlation
})

simple_score_df["score"] = simple_score_df["abs_corr_with_price"] - simple_score_df["nan_ratio"]
simple_score_df = simple_score_df.sort_values("score", ascending=False)

print("Feature ranking by nan-ratio and correlation:")
display(simple_score_df)

top_10_simple_features = simple_score_df.head(10).index.tolist()

print("Top 10 features by nan-ratio and correlation:")
print(top_10_simple_features)

Feature ranking by nan-ratio and correlation:


,nan_ratio,abs_corr_with_price,score
bathrooms,0.0,0.168800,0.168800
bedrooms,0.0,0.117016,0.117016
Doorman,0.0,0.068461,0.068461
LaundryinUnit,0.0,0.058185,0.058185
DiningRoom,0.0,0.051676,0.051676
Elevator,0.0,0.043386,0.043386
FitnessCenter,0.0,0.041127,0.041127
Dishwasher,0.0,0.038111,0.038111
Terrace,0.0,0.037136,0.037136
OutdoorSpace,0.0,0.031051,0.031051


Top 10 features by nan-ratio and correlation:
['bathrooms', 'bedrooms', 'Doorman', 'LaundryinUnit', 'DiningRoom', 'Elevator', 'FitnessCenter', 'Dishwasher', 'Terrace', 'OutdoorSpace']


## Ranking features with a simple filter method

In this step, I combine two simple signals for feature quality: low missing-value ratio and high correlation with the target.  
This gives a lightweight baseline feature selection method before moving to more advanced approaches.

In [26]:
X_train_simple = train_df_fs[top_10_simple_features]
X_val_simple = val_df_fs[top_10_simple_features]
X_test_simple = test_df_fs[top_10_simple_features]

scaler_simple = StandardScaler()

X_train_simple_scaled = scaler_simple.fit_transform(X_train_simple)
X_val_simple_scaled = scaler_simple.transform(X_val_simple)
X_test_simple_scaled = scaler_simple.transform(X_test_simple)

lasso_simple = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso_simple.fit(X_train_simple_scaled, y_train)

val_pred_simple = lasso_simple.predict(X_val_simple_scaled)

print("Validation MAE with simple top 10 features:", mean_absolute_error(y_val, val_pred_simple))
print("Validation MSE with simple top 10 features:", mean_squared_error(y_val, val_pred_simple))

Validation MAE with simple top 10 features: 924.6757037029549
Validation MSE with simple top 10 features: 4031983.04887121


## Evaluating the simple top 10 feature subset

After selecting the top 10 features with the simple filter method, I retrain the model and compare its validation performance with the previous Lasso-based subset.

This result shows how a simple filter-based method compares with Lasso-based feature selection.  
If the validation metrics stay close, it suggests that even a lightweight feature ranking strategy can preserve much of the useful signal.

## Custom permutation importance

In this step, I implement permutation importance manually using the validation set.  
The idea is to measure how much the validation MAE worsens when the values of one feature are randomly shuffled.  
If the MAE increases a lot, that feature is likely important for the model.

In [27]:
def permutation_importance_mae(model, X_val_scaled, y_val, feature_names, random_state=42):
    np.random.seed(random_state)

    baseline_pred = model.predict(X_val_scaled)
    baseline_mae = mean_absolute_error(y_val, baseline_pred)

    importance_scores = {}

    for i, feature in enumerate(feature_names):
        X_permuted = X_val_scaled.copy()
        shuffled_values = X_permuted[:, i].copy()
        np.random.shuffle(shuffled_values)
        X_permuted[:, i] = shuffled_values

        permuted_pred = model.predict(X_permuted)
        permuted_mae = mean_absolute_error(y_val, permuted_pred)

        importance_scores[feature] = permuted_mae - baseline_mae

    return pd.Series(importance_scores).sort_values(ascending=False)


perm_importance = permutation_importance_mae(
    lasso,
    X_val_scaled,
    y_val,
    feature_list,
    random_state=42
)

print("Permutation importance ranking:")
display(perm_importance)

top_10_perm_features = perm_importance.head(10).index.tolist()

print("Top 10 features selected by permutation importance:")
print(top_10_perm_features)

Permutation importance ranking:


bathrooms            472.227116
Doorman              198.451920
bedrooms             149.150867
LaundryinUnit         33.150622
LaundryinBuilding     19.767783
DogsAllowed           15.070507
CatsAllowed           10.944259
Elevator               9.376727
NoFee                  8.076785
Terrace                6.793307
LaundryInBuilding      6.179848
HighSpeedInternet      5.675020
OutdoorSpace           2.376931
Balcony                1.771584
HardwoodFloors         1.369478
DiningRoom             1.207292
Dishwasher             0.005462
NewConstruction       -0.049776
RoofDeck              -0.182878
SwimmingPool          -0.198594
FitnessCenter         -0.200126
Pre-War               -1.445341
dtype: float64

Top 10 features selected by permutation importance:
['bathrooms', 'Doorman', 'bedrooms', 'LaundryinUnit', 'LaundryinBuilding', 'DogsAllowed', 'CatsAllowed', 'Elevator', 'NoFee', 'Terrace']


## Evaluating the top 10 permutation features

After selecting the top 10 features using permutation importance, I retrain the model on this reduced subset and evaluate it on the validation set.  
This helps me check whether these features preserve most of the useful predictive information.

In [28]:
X_train_perm = train_df_fs[top_10_perm_features]
X_val_perm = val_df_fs[top_10_perm_features]

scaler_perm = StandardScaler()

X_train_perm_scaled = scaler_perm.fit_transform(X_train_perm)
X_val_perm_scaled = scaler_perm.transform(X_val_perm)

lasso_perm = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso_perm.fit(X_train_perm_scaled, y_train)

val_pred_perm = lasso_perm.predict(X_val_perm_scaled)

print("Validation MAE with permutation importance top 10:", mean_absolute_error(y_val, val_pred_perm))
print("Validation MSE with permutation importance top 10:", mean_squared_error(y_val, val_pred_perm))

Validation MAE with permutation importance top 10: 914.2995112340003
Validation MSE with permutation importance top 10: 3986157.4656519396


## SHAP-based feature importance

In this step, I use SHAP to estimate feature importance from the model predictions.  
Unlike simple coefficient-based ranking, SHAP provides a more detailed view of how each feature contributes to the output.

In [29]:
import sys
!{sys.executable} -m pip install shap

In [30]:
import shap

explainer = shap.Explainer(lasso, X_train_scaled)
shap_values = explainer(X_val_scaled)

shap_importance = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=feature_list
).sort_values(ascending=False)

print("SHAP feature importance ranking:")
display(shap_importance)

top_10_shap_features = shap_importance.head(10).index.tolist()

print("Top 10 features selected by SHAP:")
print(top_10_shap_features)

SHAP feature importance ranking:


bathrooms            777.847129
Doorman              560.359240
bedrooms             381.100651
LaundryinUnit        181.869655
DogsAllowed          181.209460
LaundryinBuilding    171.874555
NoFee                124.838726
HardwoodFloors       105.009316
Elevator              96.438233
CatsAllowed           78.731082
LaundryInBuilding     69.441621
Pre-War               57.127181
HighSpeedInternet     55.760150
FitnessCenter         40.290210
Terrace               37.902767
DiningRoom            30.005899
OutdoorSpace          29.054274
Balcony               27.239833
RoofDeck              18.517659
SwimmingPool           2.159729
NewConstruction        1.144615
Dishwasher             0.070150
dtype: float64

Top 10 features selected by SHAP:
['bathrooms', 'Doorman', 'bedrooms', 'LaundryinUnit', 'DogsAllowed', 'LaundryinBuilding', 'NoFee', 'HardwoodFloors', 'Elevator', 'CatsAllowed']


## Evaluating the top 10 SHAP features

After selecting the top 10 features with SHAP, I retrain the model on this reduced subset and evaluate it on the validation set.  

This helps me compare the SHAP-based feature subset with the previous selection methods.

In [31]:
X_train_shap = train_df_fs[top_10_shap_features]
X_val_shap = val_df_fs[top_10_shap_features]

scaler_shap = StandardScaler()

X_train_shap_scaled = scaler_shap.fit_transform(X_train_shap)
X_val_shap_scaled = scaler_shap.transform(X_val_shap)

lasso_shap = Lasso(alpha=0.01, max_iter=10000, random_state=42)
lasso_shap.fit(X_train_shap_scaled, y_train)

val_pred_shap = lasso_shap.predict(X_val_shap_scaled)

print("Validation MAE with SHAP top 10:", mean_absolute_error(y_val, val_pred_shap))
print("Validation MSE with SHAP top 10:", mean_squared_error(y_val, val_pred_shap))

Validation MAE with SHAP top 10: 912.0134243486472
Validation MSE with SHAP top 10: 3983978.050415675


This result shows whether the SHAP-based top 10 features can preserve most of the predictive information while also offering a more interpretable ranking of feature importance.

In [32]:
import shap

explainer = shap.LinearExplainer(lasso, X_train_scaled)
shap_values = explainer.shap_values(X_val_scaled)

shap_importance = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_list
).sort_values(ascending=False)

print("SHAP feature importance ranking:")
display(shap_importance)

top_10_shap_features = shap_importance.head(10).index.tolist()

print("Top 10 features selected by SHAP:")
print(top_10_shap_features)

SHAP feature importance ranking:


bathrooms            777.847129
Doorman              560.359240
bedrooms             381.100651
LaundryinUnit        181.869655
DogsAllowed          181.209460
LaundryinBuilding    171.874555
NoFee                124.838726
HardwoodFloors       105.009316
Elevator              96.438233
CatsAllowed           78.731082
LaundryInBuilding     69.441621
Pre-War               57.127181
HighSpeedInternet     55.760150
FitnessCenter         40.290210
Terrace               37.902767
DiningRoom            30.005899
OutdoorSpace          29.054274
Balcony               27.239833
RoofDeck              18.517659
SwimmingPool           2.159729
NewConstruction        1.144615
Dishwasher             0.070150
dtype: float64

Top 10 features selected by SHAP:
['bathrooms', 'Doorman', 'bedrooms', 'LaundryinUnit', 'DogsAllowed', 'LaundryinBuilding', 'NoFee', 'HardwoodFloors', 'Elevator', 'CatsAllowed']


## Comparing feature selection methods

Finally, I compare the different feature selection methods using validation metrics.  
This helps me evaluate which method gives the best trade-off between compactness, interpretability, and predictive performance.

In [33]:
feature_selection_results = pd.DataFrame({
    "method": [
        "All features",
        "Lasso top 10",
        "Nan-ratio + correlation top 10",
        "Permutation importance top 10",
        "SHAP top 10"
    ],
    "num_features": [
        len(feature_list),
        len(top_10_lasso_features),
        len(top_10_simple_features),
        len(top_10_perm_features),
        len(top_10_shap_features)
    ],
    "validation_mae": [
        mean_absolute_error(y_val, val_pred),
        mean_absolute_error(y_val, val_pred_top10),
        mean_absolute_error(y_val, val_pred_simple),
        mean_absolute_error(y_val, val_pred_perm),
        mean_absolute_error(y_val, val_pred_shap)
    ],
    "validation_mse": [
        mean_squared_error(y_val, val_pred),
        mean_squared_error(y_val, val_pred_top10),
        mean_squared_error(y_val, val_pred_simple),
        mean_squared_error(y_val, val_pred_perm),
        mean_squared_error(y_val, val_pred_shap)
    ],
    "speed_comment": [
        "baseline",
        "fast",
        "very fast",
        "medium",
        "slow"
    ],
    "stability_comment": [
        "baseline",
        "good",
        "medium",
        "good",
        "good"
    ]
})

feature_selection_results.sort_values("validation_mae")

,method,num_features,validation_mae,validation_mse,speed_comment,stability_comment
4,SHAP top 10,10,912.013424,3.983978e+06,slow,good
3,Permutation importance top 10,10,914.299511,3.986157e+06,medium,good
1,Lasso top 10,10,918.233273,3.997489e+06,fast,good
0,All features,22,922.213813,3.999292e+06,baseline,baseline
2,Nan-ratio + correlation top 10,10,924.675704,4.031983e+06,very fast,medium


## Feature selection conclusion

The feature selection methods produced reduced subsets with comparable validation quality.  
Lasso was simple and effective, the nan-ratio plus correlation method served as a lightweight baseline, permutation importance provided a model-aware ranking, and SHAP offered the richest interpretability.

Overall, the best method depends on the priority: simplicity and speed, model-aware ranking, or deeper explanation.

## Preparing the data for hyperparameter optimization

At this stage, I prepare the train, validation, and test sets for ElasticNet.  
I keep the same time-based split chosen earlier, and I standardize the features because ElasticNet is sensitive to feature scale.

In [34]:
from sklearn.linear_model import ElasticNet

X_train_hp = train_df_fs[feature_list]
X_val_hp = val_df_fs[feature_list]
X_test_hp = test_df_fs[feature_list]

y_train_hp = train_df_fs["price"]
y_val_hp = val_df_fs["price"]
y_test_hp = test_df_fs["price"]

scaler_hp = StandardScaler()

X_train_hp_scaled = scaler_hp.fit_transform(X_train_hp)
X_val_hp_scaled = scaler_hp.transform(X_val_hp)
X_test_hp_scaled = scaler_hp.transform(X_test_hp)

## Manual Grid Search with ElasticNet

I start hyperparameter optimization with a manual Grid Search.  
I test a fixed set of `alpha` and `l1_ratio` values, train an ElasticNet model for each combination, and evaluate the result on the validation set.

In [35]:
alpha_grid = [0.001, 0.01, 0.1, 1.0]
l1_ratio_grid = [0.2, 0.5, 0.8]

grid_results = []

for alpha in alpha_grid:
    for l1_ratio in l1_ratio_grid:
        model = ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=10000,
            random_state=42
        )
        
        model.fit(X_train_hp_scaled, y_train_hp)
        val_pred_hp = model.predict(X_val_hp_scaled)
        
        mae = mean_absolute_error(y_val_hp, val_pred_hp)
        mse = mean_squared_error(y_val_hp, val_pred_hp)

        grid_results.append({
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "validation_mae": mae,
            "validation_mse": mse
        })

grid_results_df = pd.DataFrame(grid_results).sort_values("validation_mae")
grid_results_df

,alpha,l1_ratio,validation_mae,validation_mse
11,1.000,0.8,891.820960,4.062104e+06
10,1.000,0.5,895.975465,4.247157e+06
6,0.100,0.2,903.840240,4.010939e+06
7,0.100,0.5,909.357754,4.003504e+06
8,0.100,0.8,916.401520,3.999418e+06
9,1.000,0.2,917.210132,4.441758e+06
3,0.010,0.2,919.798226,3.999081e+06
4,0.010,0.5,920.680284,3.999110e+06
5,0.010,0.8,921.591828,3.999198e+06
0,0.001,0.2,921.973114,3.999261e+06


## Best Grid Search configuration

After evaluating all tested parameter combinations, I select the one with the lowest validation MAE as the best Grid Search result.

In [36]:
best_grid_row = grid_results_df.iloc[0]

best_grid_alpha = best_grid_row["alpha"]
best_grid_l1_ratio = best_grid_row["l1_ratio"]

print("Best Grid Search parameters:")
print("alpha =", best_grid_alpha)
print("l1_ratio =", best_grid_l1_ratio)
print("validation_mae =", best_grid_row["validation_mae"])
print("validation_mse =", best_grid_row["validation_mse"])

Best Grid Search parameters:
alpha = 1.0
l1_ratio = 0.8
validation_mae = 891.8209595391938
validation_mse = 4062103.7482651575


## Training and evaluating the best Grid Search model

After selecting the best hyperparameter combination from Grid Search, I retrain ElasticNet with these values and evaluate it on both the validation and test sets.

In [37]:
best_grid_model = ElasticNet(
    alpha=best_grid_alpha,
    l1_ratio=best_grid_l1_ratio,
    max_iter=10000,
    random_state=42
)

best_grid_model.fit(X_train_hp_scaled, y_train_hp)

val_pred_grid = best_grid_model.predict(X_val_hp_scaled)
test_pred_grid = best_grid_model.predict(X_test_hp_scaled)

print("Best Grid Search Validation MAE:", mean_absolute_error(y_val_hp, val_pred_grid))
print("Best Grid Search Validation MSE:", mean_squared_error(y_val_hp, val_pred_grid))

print("Best Grid Search Test MAE:", mean_absolute_error(y_test_hp, test_pred_grid))
print("Best Grid Search Test MSE:", mean_squared_error(y_test_hp, test_pred_grid))

Best Grid Search Validation MAE: 891.8209595391938
Best Grid Search Validation MSE: 4062103.7482651575
Best Grid Search Test MAE: 1433.1113904192969
Best Grid Search Test MSE: 2173847400.1825395


This step gives the final validation and test performance of the best Grid Search configuration, which provides a stronger estimate of how well the model generalizes.

## Random Search with ElasticNet

After Grid Search, I also test Random Search.  
Instead of evaluating every possible combination, I randomly sample a smaller number of parameter combinations and compare their validation performance.

In [38]:
np.random.seed(42)

alpha_candidates = [0.001, 0.01, 0.1, 1.0]
l1_ratio_candidates = [0.2, 0.5, 0.8]

random_results = []
n_trials = 6

sampled_combinations = []

while len(sampled_combinations) < n_trials:
    combo = (
        np.random.choice(alpha_candidates),
        np.random.choice(l1_ratio_candidates)
    )
    if combo not in sampled_combinations:
        sampled_combinations.append(combo)

for alpha, l1_ratio in sampled_combinations:
    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=10000,
        random_state=42
    )

    model.fit(X_train_hp_scaled, y_train_hp)
    val_pred_random = model.predict(X_val_hp_scaled)

    mae = mean_absolute_error(y_val_hp, val_pred_random)
    mse = mean_squared_error(y_val_hp, val_pred_random)

    random_results.append({
        "alpha": alpha,
        "l1_ratio": l1_ratio,
        "validation_mae": mae,
        "validation_mse": mse
    })

random_results_df = pd.DataFrame(random_results).sort_values("validation_mae")
random_results_df

,alpha,l1_ratio,validation_mae,validation_mse
5,1.000,0.8,891.820960,4.062104e+06
0,0.100,0.2,903.840240,4.010939e+06
1,0.100,0.8,916.401520,3.999418e+06
2,1.000,0.2,917.210132,4.441758e+06
4,0.010,0.8,921.591828,3.999198e+06
3,0.001,0.8,922.160505,3.999291e+06


## Best Random Search configuration

After evaluating the sampled parameter combinations, I select the one with the lowest validation MAE as the best Random Search result.

In [39]:
best_random_row = random_results_df.iloc[0]

best_random_alpha = best_random_row["alpha"]
best_random_l1_ratio = best_random_row["l1_ratio"]

print("Best Random Search parameters:")
print("alpha =", best_random_alpha)
print("l1_ratio =", best_random_l1_ratio)
print("validation_mae =", best_random_row["validation_mae"])
print("validation_mse =", best_random_row["validation_mse"])

Best Random Search parameters:
alpha = 1.0
l1_ratio = 0.8
validation_mae = 891.8209595391938
validation_mse = 4062103.7482651575


## Training and evaluating the best Random Search model

After selecting the best parameter combination from Random Search, I retrain ElasticNet with these values and evaluate it on both the validation and test sets.

In [40]:
best_random_model = ElasticNet(
    alpha=best_random_alpha,
    l1_ratio=best_random_l1_ratio,
    max_iter=10000,
    random_state=42
)

best_random_model.fit(X_train_hp_scaled, y_train_hp)

val_pred_random_best = best_random_model.predict(X_val_hp_scaled)
test_pred_random_best = best_random_model.predict(X_test_hp_scaled)

print("Best Random Search Validation MAE:", mean_absolute_error(y_val_hp, val_pred_random_best))
print("Best Random Search Validation MSE:", mean_squared_error(y_val_hp, val_pred_random_best))

print("Best Random Search Test MAE:", mean_absolute_error(y_test_hp, test_pred_random_best))
print("Best Random Search Test MSE:", mean_squared_error(y_test_hp, test_pred_random_best))

Best Random Search Validation MAE: 891.8209595391938
Best Random Search Validation MSE: 4062103.7482651575
Best Random Search Test MAE: 1433.1113904192969
Best Random Search Test MSE: 2173847400.1825395


This step shows the final validation and test performance of the best configuration found by Random Search, which makes it possible to compare it directly with the best Grid Search model.

## Comparing Grid Search and Random Search

After evaluating the best model from each tuning method, I compare their selected hyperparameters and their validation and test performance.  
This helps me understand whether Random Search can achieve results close to Grid Search with fewer trials.

In [41]:
tuning_comparison_df = pd.DataFrame({
    "method": ["Grid Search", "Random Search"],
    "best_alpha": [best_grid_alpha, best_random_alpha],
    "best_l1_ratio": [best_grid_l1_ratio, best_random_l1_ratio],
    "validation_mae": [
        mean_absolute_error(y_val_hp, val_pred_grid),
        mean_absolute_error(y_val_hp, val_pred_random_best)
    ],
    "validation_mse": [
        mean_squared_error(y_val_hp, val_pred_grid),
        mean_squared_error(y_val_hp, val_pred_random_best)
    ],
    "test_mae": [
        mean_absolute_error(y_test_hp, test_pred_grid),
        mean_absolute_error(y_test_hp, test_pred_random_best)
    ],
    "test_mse": [
        mean_squared_error(y_test_hp, test_pred_grid),
        mean_squared_error(y_test_hp, test_pred_random_best)
    ]
})

tuning_comparison_df

,method,best_alpha,best_l1_ratio,validation_mae,validation_mse,test_mae,test_mse
0,Grid Search,1.0,0.8,891.82096,4.062104e+06,1433.11139,2.173847e+09
1,Random Search,1.0,0.8,891.82096,4.062104e+06,1433.11139,2.173847e+09


In [42]:
import sys
!{sys.executable} -m pip install optuna

In [43]:
import optuna
print(optuna.__version__)

4.8.0


## Hyperparameter optimization with Optuna

After trying Grid Search and Random Search, I also use Optuna as a more adaptive optimization method.  
Instead of testing a fixed grid or a random subset, Optuna searches the parameter space more intelligently based on the results of previous trials.

In [44]:
def objective(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.2, 0.8)

    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=10000,
        random_state=42
    )

    model.fit(X_train_hp_scaled, y_train_hp)
    val_pred = model.predict(X_val_hp_scaled)

    mae = mean_absolute_error(y_val_hp, val_pred)
    return mae


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print("Best Optuna parameters:", study.best_params)
print("Best Optuna validation MAE:", study.best_value)

[I 2026-05-04 07:19:52,684] A new study created in memory with name: no-name-e84b4fcb-a19f-4f8f-a6cf-e8b2868f1d94
[I 2026-05-04 07:19:52,723] Trial 0 finished with value: 891.1138148253023 and parameters: {'alpha': 0.9160024034163724, 'l1_ratio': 0.5776433026578776}. Best is trial 0 with value: 891.1138148253023.
[I 2026-05-04 07:19:52,788] Trial 1 finished with value: 922.0630741700766 and parameters: {'alpha': 0.001458959177525299, 'l1_ratio': 0.6503178966488007}. Best is trial 0 with value: 891.1138148253023.
[I 2026-05-04 07:19:52,838] Trial 2 finished with value: 919.2559661216153 and parameters: {'alpha': 0.021043655927792684, 'l1_ratio': 0.5325496550630731}. Best is trial 0 with value: 891.1138148253023.
[I 2026-05-04 07:19:52,877] Trial 3 finished with value: 903.512107905242 and parameters: {'alpha': 0.19519646240792965, 'l1_ratio': 0.581239405335081}. Best is trial 0 with value: 891.1138148253023.
[I 2026-05-04 07:19:53,025] Trial 4 finished with value: 918.8834314303241 and 

Best Optuna parameters: {'alpha': 0.41885371565094937, 'l1_ratio': 0.332336621498006}
Best Optuna validation MAE: 889.7942451671667


###  Optuna objective

In the objective function, I let Optuna choose `alpha` and `l1_ratio`, train an ElasticNet model with these values, and evaluate it using validation MAE.  
The goal is to minimize the validation MAE.

## Training and evaluating the best Optuna model

After obtaining the best parameter combination from Optuna, I retrain ElasticNet with these values and evaluate it on both the validation and test sets.

In [45]:
best_optuna_alpha = study.best_params["alpha"]
best_optuna_l1_ratio = study.best_params["l1_ratio"]

best_optuna_model = ElasticNet(
    alpha=best_optuna_alpha,
    l1_ratio=best_optuna_l1_ratio,
    max_iter=10000,
    random_state=42
)

best_optuna_model.fit(X_train_hp_scaled, y_train_hp)

val_pred_optuna = best_optuna_model.predict(X_val_hp_scaled)
test_pred_optuna = best_optuna_model.predict(X_test_hp_scaled)

print("Best Optuna Validation MAE:", mean_absolute_error(y_val_hp, val_pred_optuna))
print("Best Optuna Validation MSE:", mean_squared_error(y_val_hp, val_pred_optuna))

print("Best Optuna Test MAE:", mean_absolute_error(y_test_hp, test_pred_optuna))
print("Best Optuna Test MSE:", mean_squared_error(y_test_hp, test_pred_optuna))

Best Optuna Validation MAE: 889.7942451671667
Best Optuna Validation MSE: 4106452.0515972604
Best Optuna Test MAE: 1431.2955831290187
Best Optuna Test MSE: 2173937468.4564385


This step gives the final validation and test performance of the Optuna-selected model, which makes it possible to compare Optuna directly with Grid Search and Random Search.

## Final comparison of tuning methods

At the end of the tuning stage, I compare the best results from Grid Search, Random Search, and Optuna using both validation and test metrics.  
Sorting by test MAE helps me see which method produced the strongest final generalization performance.

In [46]:
final_tuning_comparison_df = pd.DataFrame({
    "method": ["Grid Search", "Random Search", "Optuna"],
    "best_alpha": [
        best_grid_alpha,
        best_random_alpha,
        best_optuna_alpha
    ],
    "best_l1_ratio": [
        best_grid_l1_ratio,
        best_random_l1_ratio,
        best_optuna_l1_ratio
    ],
    "validation_mae": [
        mean_absolute_error(y_val_hp, val_pred_grid),
        mean_absolute_error(y_val_hp, val_pred_random_best),
        mean_absolute_error(y_val_hp, val_pred_optuna)
    ],
    "validation_mse": [
        mean_squared_error(y_val_hp, val_pred_grid),
        mean_squared_error(y_val_hp, val_pred_random_best),
        mean_squared_error(y_val_hp, val_pred_optuna)
    ],
    "test_mae": [
        mean_absolute_error(y_test_hp, test_pred_grid),
        mean_absolute_error(y_test_hp, test_pred_random_best),
        mean_absolute_error(y_test_hp, test_pred_optuna)
    ],
    "test_mse": [
        mean_squared_error(y_test_hp, test_pred_grid),
        mean_squared_error(y_test_hp, test_pred_random_best),
        mean_squared_error(y_test_hp, test_pred_optuna)
    ]
})

final_tuning_comparison_df.sort_values("test_mae")

,method,best_alpha,best_l1_ratio,validation_mae,validation_mse,test_mae,test_mse
2,Optuna,0.418854,0.332337,889.794245,4.106452e+06,1431.295583,2.173937e+09
0,Grid Search,1.000000,0.800000,891.820960,4.062104e+06,1433.111390,2.173847e+09
1,Random Search,1.000000,0.800000,891.820960,4.062104e+06,1433.111390,2.173847e+09


## Tuning conclusion

The final comparison shows how the three tuning methods differ in their selected parameters and predictive performance.  
By sorting the results by test MAE, I can identify which method achieved the strongest generalization on unseen data.

In practice, Grid Search is systematic, Random Search is lighter, and Optuna is more adaptive.  
The most suitable method depends on the balance between search cost and final model quality.

## Optuna with cross-validation

As a final tuning step, I use Optuna together with cross-validation.  
Instead of evaluating each trial on a single validation split, I evaluate it across multiple folds and optimize the mean validation MAE.

In [47]:
from sklearn.model_selection import KFold

def objective_cv(trial):
    alpha = trial.suggest_float("alpha", 0.001, 1.0, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.2, 0.8)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_maes = []

    for train_idx, val_idx in kf.split(model_df):
        train_fold = model_df.iloc[train_idx]
        val_fold = model_df.iloc[val_idx]

        X_train_fold = train_fold[feature_list]
        y_train_fold = train_fold["price"]

        X_val_fold = val_fold[feature_list]
        y_val_fold = val_fold["price"]

        scaler_fold = StandardScaler()

        X_train_fold_scaled = scaler_fold.fit_transform(X_train_fold)
        X_val_fold_scaled = scaler_fold.transform(X_val_fold)

        model = ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            max_iter=10000,
            random_state=42
        )

        model.fit(X_train_fold_scaled, y_train_fold)
        val_pred_fold = model.predict(X_val_fold_scaled)

        fold_mae = mean_absolute_error(y_val_fold, val_pred_fold)
        fold_maes.append(fold_mae)

    return np.mean(fold_maes)


study_cv = optuna.create_study(direction="minimize")
study_cv.optimize(objective_cv, n_trials=20)

print("Best Optuna CV parameters:", study_cv.best_params)
print("Best Optuna CV mean MAE:", study_cv.best_value)

[I 2026-05-04 07:19:54,678] A new study created in memory with name: no-name-469c9105-ec58-46d0-8013-66acbe543ae5
[I 2026-05-04 07:19:55,228] Trial 0 finished with value: 1161.480053310081 and parameters: {'alpha': 0.001643552603629332, 'l1_ratio': 0.5825932362298142}. Best is trial 0 with value: 1161.480053310081.
[I 2026-05-04 07:19:55,798] Trial 1 finished with value: 1154.9859830298597 and parameters: {'alpha': 0.017460594962823036, 'l1_ratio': 0.2737269258652435}. Best is trial 1 with value: 1154.9859830298597.
[I 2026-05-04 07:19:56,174] Trial 2 finished with value: 1154.2265004715803 and parameters: {'alpha': 0.05839521529035047, 'l1_ratio': 0.759457847630961}. Best is trial 2 with value: 1154.2265004715803.
[I 2026-05-04 07:19:56,716] Trial 3 finished with value: 1161.7085231251208 and parameters: {'alpha': 0.0013414909898189996, 'l1_ratio': 0.7884813275507765}. Best is trial 2 with value: 1154.2265004715803.
[I 2026-05-04 07:19:57,195] Trial 4 finished with value: 1161.6728981

Best Optuna CV parameters: {'alpha': 0.8744350232363246, 'l1_ratio': 0.44551527836283866}
Best Optuna CV mean MAE: 1076.9577502290563


##  cross-validation objective

In this objective function, Optuna chooses `alpha` and `l1_ratio`, then evaluates ElasticNet with 5-fold cross-validation.  
For each fold, the model is trained on the training fold and evaluated on the validation fold.  
The final score is the mean MAE across all folds.

## Training and evaluating the best Optuna CV model

After obtaining the best parameter combination from Optuna with cross-validation, I retrain ElasticNet with these values on the training set and evaluate it on the validation and test sets.

In [48]:
best_optuna_cv_alpha = study_cv.best_params["alpha"]
best_optuna_cv_l1_ratio = study_cv.best_params["l1_ratio"]

best_optuna_cv_model = ElasticNet(
    alpha=best_optuna_cv_alpha,
    l1_ratio=best_optuna_cv_l1_ratio,
    max_iter=10000,
    random_state=42
)

best_optuna_cv_model.fit(X_train_hp_scaled, y_train_hp)

val_pred_optuna_cv = best_optuna_cv_model.predict(X_val_hp_scaled)
test_pred_optuna_cv = best_optuna_cv_model.predict(X_test_hp_scaled)

print("Optuna CV Validation MAE:", mean_absolute_error(y_val_hp, val_pred_optuna_cv))
print("Optuna CV Validation MSE:", mean_squared_error(y_val_hp, val_pred_optuna_cv))

print("Optuna CV Test MAE:", mean_absolute_error(y_test_hp, test_pred_optuna_cv))
print("Optuna CV Test MSE:", mean_squared_error(y_test_hp, test_pred_optuna_cv))

Optuna CV Validation MAE: 895.1802170017779
Optuna CV Validation MSE: 4237064.722382881
Optuna CV Test MAE: 1437.211048384523
Optuna CV Test MSE: 2174162885.151322


This step provides the final validation and test performance of the model selected by Optuna with cross-validation, which makes it possible to compare it with the previous tuning approaches.

## Extended comparison of tuning methods

At the end of the tuning stage, I compare the best results from Grid Search, Random Search, Optuna, and Optuna with K-Fold cross-validation.  
Sorting the table by test MAE helps identify which method produced the strongest final generalization performance.

In [49]:
extended_tuning_comparison_df = pd.DataFrame({
    "method": ["Grid Search", "Random Search", "Optuna", "Optuna + KFold CV"],
    "best_alpha": [
        best_grid_alpha,
        best_random_alpha,
        best_optuna_alpha,
        best_optuna_cv_alpha
    ],
    "best_l1_ratio": [
        best_grid_l1_ratio,
        best_random_l1_ratio,
        best_optuna_l1_ratio,
        best_optuna_cv_l1_ratio
    ],
    "validation_mae": [
        mean_absolute_error(y_val_hp, val_pred_grid),
        mean_absolute_error(y_val_hp, val_pred_random_best),
        mean_absolute_error(y_val_hp, val_pred_optuna),
        mean_absolute_error(y_val_hp, val_pred_optuna_cv)
    ],
    "validation_mse": [
        mean_squared_error(y_val_hp, val_pred_grid),
        mean_squared_error(y_val_hp, val_pred_random_best),
        mean_squared_error(y_val_hp, val_pred_optuna),
        mean_squared_error(y_val_hp, val_pred_optuna_cv)
    ],
    "test_mae": [
        mean_absolute_error(y_test_hp, test_pred_grid),
        mean_absolute_error(y_test_hp, test_pred_random_best),
        mean_absolute_error(y_test_hp, test_pred_optuna),
        mean_absolute_error(y_test_hp, test_pred_optuna_cv)
    ],
    "test_mse": [
        mean_squared_error(y_test_hp, test_pred_grid),
        mean_squared_error(y_test_hp, test_pred_random_best),
        mean_squared_error(y_test_hp, test_pred_optuna),
        mean_squared_error(y_test_hp, test_pred_optuna_cv)
    ]
})

extended_tuning_comparison_df.sort_values("test_mae")

,method,best_alpha,best_l1_ratio,validation_mae,validation_mse,test_mae,test_mse
2,Optuna,0.418854,0.332337,889.794245,4.106452e+06,1431.295583,2.173937e+09
0,Grid Search,1.000000,0.800000,891.820960,4.062104e+06,1433.111390,2.173847e+09
1,Random Search,1.000000,0.800000,891.820960,4.062104e+06,1433.111390,2.173847e+09
3,Optuna + KFold CV,0.874435,0.445515,895.180217,4.237065e+06,1437.211048,2.174163e+09


## Final tuning conclusion

The extended comparison shows that all tuning methods can produce competitive results, but they differ in search strategy and stability.

- Grid Search is systematic and easy to interpret, but it can be expensive.
- Random Search is more lightweight and can still find strong parameter combinations.
- Optuna is more adaptive and can search the parameter space more efficiently.
- Optuna with K-Fold cross-validation provides a more stable selection process because each trial is evaluated across multiple folds.

Overall, the most suitable tuning method depends on the balance between computational cost, search efficiency, and robustness of the final result.

## Final project summary

At the end of the project, I summarize the main choices made in each stage: the validation strategy, the feature selection method, and the hyperparameter tuning approach.  
This provides a compact overview of the final pipeline and the reasoning behind each selected component.

In [50]:
final_summary_df = pd.DataFrame({
    "component": [
        "Validation scheme",
        "Feature selection",
        "Hyperparameter tuning"
    ],
    "selected_choice": [
        "Time-aware validation / TimeSeries-style logic",
        "Permutation importance top 10",
        "Optuna"
    ],
    "reason": [
        "Most realistic when time ordering matters and helps avoid future leakage.",
        "Best validation MAE among compact feature subsets with only 10 features.",
        "Best holdout validation and test MAE among the tuning methods."
    ]
})

final_summary_df

,component,selected_choice,reason
0,Validation scheme,Time-aware validation / TimeSeries-style logic,Most realistic when time ordering matters and ...
1,Feature selection,Permutation importance top 10,Best validation MAE among compact feature subs...
2,Hyperparameter tuning,Optuna,Best holdout validation and test MAE among the...


## Final conclusion

In the end, the most suitable pipeline combined time-aware validation, permutation-based feature selection, and Optuna tuning.  
This combination gave the most realistic evaluation setup, kept the feature set compact, and achieved the strongest predictive performance on unseen data.